In [1]:
import os
import time
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
# from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ART

from art.attacks.evasion import ProjectedGradientDescent, FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier, SklearnClassifier, XGBoostClassifier, LightGBMClassifier, CatBoostARTClassifier, KerasClassifier

I0000 00:00:1778006577.903013 2945720 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778006577.950226 2945720 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778006579.123044 2945720 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packa

In [2]:
# Configuración de visualización

sns.set_style("whitegrid")

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
TRAIN_DEVICE = "/GPU:0" if HAS_GPU else "/CPU:0"
INFER_DEVICE = "/CPU:0"

if HAS_GPU:
    print("GPU detectada. Se entrenarán los modelos compatibles en GPU y se medirá la inferencia en CPU cuando aplique.")
else:
    print("No hay GPU disponible. Todo el notebook se ejecutará en CPU.")

tf.keras.backend.clear_session()

def build_mlp_model(input_dim, hidden_units):
    model = keras.Sequential()
    model.add(keras.layers.InputLayer(input_shape=(input_dim,)))
    for units in hidden_units:
        model.add(keras.layers.Dense(units, activation="relu"))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


DEFAULT_CNN_DROPOUT = 0.2

def build_cnn1d_model(n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_CNN_DROPOUT):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features, 1)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding="same", activation="relu"),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def clone_keras_model_to_cpu(builder_fn, trained_model, *builder_args):
    with tf.device(INFER_DEVICE):
        cpu_model = builder_fn(*builder_args)
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model

GPU detectada. Se entrenarán los modelos compatibles en GPU y se medirá la inferencia en CPU cuando aplique.


In [3]:
# ==========================================
# 1. CARGA DE DATOS
# ==========================================

TARGET_COL = "label"
prepared_path = "../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv"

df_encoded = pl.read_csv(prepared_path)

feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy()

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.2,
    random_state=42,
    stratify=y_np[train_full_idx],
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]
y_full_train = y_np[train_full_idx]
y_test_np = y_np[test_idx]

X_train_np = X_np[train_idx]
X_val_np = X_np[val_idx]
y_train_np = y_np[train_idx]
y_val_np = y_np[val_idx]

print(f"Entrenamiento: {len(X_train_np):,} muestras")
print(f"Validación:    {len(X_val_np):,} muestras")
print(f"Test:          {len(X_test_np):,} muestras")
print(f"Clases en test: {np.unique(y_test_np)}")
print(f"Total muestras: {len(X_np):,}")
print("La clase 0 corresponde a Benign y la clase 1 a Malicious.")


Entrenamiento: 745,504 muestras
Validación:    186,376 muestras
Test:          232,971 muestras
Clases en test: [0 1]
Total muestras: 1,164,851
La clase 0 corresponde a Benign y la clase 1 a Malicious.


In [4]:
# ==========================================
# 2. CONFIGURACIÓN DE LOS GANADORES
# ==========================================

RF_CONFIG = {"n_estimators": 50, "max_depth": 21}
XGB_CONFIG = {"n_estimators": 50, "max_depth": 6, "learning_rate": 0.1}
LGBM_CONFIG = {"n_estimators": 50, "num_leaves": 15, "max_depth": 6, "learning_rate": 0.1}
CATBOOST_CONFIG = {"iterations": 350, "depth": 3, "learning_rate": 0.1}

SVM_C = 0.801714
MLP_INPUT_DIM = X_full_train_np.shape[1]
MLP_HIDDEN_UNITS = (32, 48, 48)
CNN1D_CONFIG = {"nf": 64, "k": 4, "d": 80}

In [5]:
# ==========================================
# 3. ENTRENAMIENTO, BENCHMARK Y MÉTRICAS
# ==========================================

X_test_np_arr = np.array(X_test_np)

print("Entrenando Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=RF_CONFIG["n_estimators"],
    max_depth=RF_CONFIG["max_depth"],
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)
rf_model.fit(X_full_train_np, y_full_train)

# ==========================================

print("\nEntrenando XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=XGB_CONFIG["n_estimators"],
    max_depth=XGB_CONFIG["max_depth"],
    learning_rate=XGB_CONFIG["learning_rate"],
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    device="cuda" if HAS_GPU else "cpu",
    random_state=42,
)
xgb_model.fit(X_full_train_np, y_full_train)

# ==========================================

print("\nEntrenando LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=LGBM_CONFIG["n_estimators"],
    num_leaves=LGBM_CONFIG["num_leaves"],
    max_depth=LGBM_CONFIG["max_depth"],
    learning_rate=LGBM_CONFIG["learning_rate"],
    objective="binary",
    device_type="gpu" if HAS_GPU else "cpu",
    n_jobs=-1,
    random_state=42,
    verbosity=-1
)
lgbm_model.fit(X_full_train_np, y_full_train)

# ==========================================

print("\nEntrenando CatBoost...")
cat_model = CatBoostClassifier(
    iterations=CATBOOST_CONFIG["iterations"],
    depth=CATBOOST_CONFIG["depth"],
    learning_rate=CATBOOST_CONFIG["learning_rate"],
    random_seed=42,
    logging_level="Silent",
    task_type="GPU" if HAS_GPU else "CPU"
)
cat_model.fit(X_full_train_np, y_full_train)

Entrenando Random Forest...

Entrenando XGBoost...

Entrenando LightGBM...

Entrenando CatBoost...


CatBoostClassifier(depth=3, iterations=350, learning_rate=0.1, logging_level='Silent', random_seed=42, task_type='GPU')

In [6]:
# ==========================================

print("\nEntrenando y evaluando Linear SVM...")
svm_model = make_pipeline(
    StandardScaler(),
    LinearSVC(C=SVM_C, dual=False, random_state=42, max_iter=2000)
)
svm_model.fit(X_full_train_np, y_full_train)

# ==========================================

print("\nEntrenando y evaluando TensorFlow MLP...")
tf.keras.backend.clear_session()
mlp_scaler = StandardScaler()
X_train_scaled_mlp = mlp_scaler.fit_transform(X_full_train_np)
X_test_scaled_mlp = mlp_scaler.transform(X_test_np_arr)
with tf.device(INFER_DEVICE):
    mlp_model = build_mlp_model(MLP_INPUT_DIM, MLP_HIDDEN_UNITS)
    mlp_early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    mlp_model.fit(
        X_train_scaled_mlp,
        y_full_train,
        validation_split=0.1,
        epochs=40,
        batch_size=2048,
        callbacks=[mlp_early],
        verbose=0
    )

# ==========================================

print("\nEntrenando y evaluando CNN-1D...")
tf.keras.backend.clear_session()
cnn_scaler = MinMaxScaler()
X_train_scaled_cnn = cnn_scaler.fit_transform(X_full_train_np)
X_test_scaled_cnn = cnn_scaler.transform(X_test_np_arr)
X_train_tabular_cnn = X_train_scaled_cnn.reshape(X_train_scaled_cnn.shape[0], X_train_scaled_cnn.shape[1], 1)
X_test_tabular_cnn = X_test_scaled_cnn.reshape(X_test_scaled_cnn.shape[0], X_test_scaled_cnn.shape[1], 1)
with tf.device(INFER_DEVICE):
    cnn_model = build_cnn1d_model(X_train_tabular_cnn.shape[1], CNN1D_CONFIG["nf"], CNN1D_CONFIG["k"], CNN1D_CONFIG["d"])
    cnn_early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    cnn_model.fit(
        X_train_tabular_cnn,
        y_full_train,
        validation_split=0.1,
        epochs=20,
        batch_size=1024,
        callbacks=[cnn_early],
        verbose=0
    )


Entrenando y evaluando Linear SVM...

Entrenando y evaluando TensorFlow MLP...


I0000 00:00:1778006600.833641 2945720 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 799 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:4a:00.0, compute capability: 8.9
I0000 00:00:1778006600.835275 2945720 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 43049 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0000:61:00.0, compute capability: 8.9
I0000 00:00:1778006600.845065 2945720 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 43049 MB memory:  -> device: 2, name: NVIDIA L40S, pci bus id: 0000:ca:00.0, compute capability: 8.9
I0000 00:00:1778006600.855813 2945720 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 43049 MB memory:  -> device: 3, name: NVIDIA L40S, pci bus id: 0000:e1:00.0, compute capability: 8.9
/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:27


Entrenando y evaluando CNN-1D...


In [8]:
# Evaluamos en test MLP

mlp_model_cpu = clone_keras_model_to_cpu(build_mlp_model, mlp_model, MLP_INPUT_DIM, MLP_HIDDEN_UNITS)
def mlp_predict_labels(X):
    with tf.device(INFER_DEVICE):
        y_prob = mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()
    return (y_prob > 0.5).astype(np.int8)
def mlp_predict_scores(X):
    with tf.device(INFER_DEVICE):
        return mlp_model_cpu.predict(X, batch_size=4096, verbose=0).ravel()

# F1-score de referencia en test para MLP
y_test_pred_mlp = mlp_predict_labels(X_test_scaled_mlp)
mlp_f1 = f1_score(y_test_np, y_test_pred_mlp)
print(f"\nMLP F1-score en test: {mlp_f1:.4f}")

/home/placivm_tfg/PLACI_TFG/.venv_tfg/lib/python3.11/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



MLP F1-score en test: 0.9962


In [17]:
# Extraemos restricciones tabulares a partir del train original
feature_names = feature_columns

X_full_train_df = pl.DataFrame(X_full_train_np, schema=feature_names)

tabular_constraints_df = pl.DataFrame({
    "feature": feature_names,
    "min": X_full_train_df.min().row(0),
    "max": X_full_train_df.max().row(0),
})

feature_mins = tabular_constraints_df["min"].to_numpy().astype(np.float32)
feature_maxs = tabular_constraints_df["max"].to_numpy().astype(np.float32)

tabular_constraints = {
    feature: {"min": float(min_val), "max": float(max_val)}
    for feature, min_val, max_val in zip(feature_names, feature_mins, feature_maxs)
}

clip_values_raw = (feature_mins, feature_maxs)

feature_mins_mlp = mlp_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_mlp = mlp_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)

feature_mins_cnn = cnn_scaler.transform(feature_mins.reshape(1, -1)).ravel().astype(np.float32)
feature_maxs_cnn = cnn_scaler.transform(feature_maxs.reshape(1, -1)).ravel().astype(np.float32)

one_hot_columns = [
    col for col in feature_names
    if col.startswith("proto_") or col.startswith("conn_state_")
]

attack_mask = np.array(
    [0.0 if col in one_hot_columns else 1.0 for col in feature_names],
    dtype=np.float32
)

# ART exige min < max en todas las columnas.
tiny = np.float32(1e-6)
feature_maxs_mlp = np.where(feature_maxs_mlp <= feature_mins_mlp, feature_mins_mlp + tiny, feature_maxs_mlp)
feature_maxs_cnn = np.where(feature_maxs_cnn <= feature_mins_cnn, feature_mins_cnn + tiny, feature_maxs_cnn)

clip_values_mlp = (feature_mins_mlp, feature_maxs_mlp)
clip_values_cnn = (feature_mins_cnn, feature_maxs_cnn)

print("Restricciones tabulares extraidas para IOT-23:")
display(tabular_constraints_df)

print("Columnas one-hot bloqueadas:")
print(one_hot_columns)


Restricciones tabulares extraidas para IOT-23:


feature,min,max
str,f64,f64
"""id.orig_p""",3.0,65394.0
"""id.resp_p""",0.0,65535.0
"""proto_icmp""",0.0,1.0
"""proto_tcp""",0.0,1.0
"""proto_udp""",0.0,1.0
…,…,…
"""conn_state_S1""",0.0,1.0
"""conn_state_S2""",0.0,1.0
"""conn_state_SF""",0.0,1.0


Columnas one-hot bloqueadas:
['proto_icmp', 'proto_tcp', 'proto_udp', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR']


In [18]:
# ==========================================
# FASE 1. ENVOLVER EL MODELO (Caja Blanca)
# ==========================================

print("Envolviendo el modelo MLP en ART con restricciones tabulares...")

clasificador_art_mlp = KerasClassifier(
    model=mlp_model_cpu,
    clip_values=clip_values_mlp,
    use_logits=False
)


Envolviendo el modelo MLP en ART con restricciones tabulares...


In [23]:
# ===================================================
# FASE 2. FUERZA DE ATAQUE POR CARACTERÍSTICA (eps)
# ===================================================

print("Configurando la fuerza del ataque (eps) por característica...")

eps_base = 0.05

feature_ranges_mlp = clip_values_mlp[1] - clip_values_mlp[0]
eps_vector = (eps_base * feature_ranges_mlp).astype(np.float32)

# Las columnas one-hot no deben modificarse
eps_vector = eps_vector * attack_mask
print(eps_vector)

# ART exige eps_step > 0 en todas las columnas
tiny_step = np.float32(1e-6)
eps_step_vector = np.where(eps_vector > 0, eps_vector / 4.0, tiny_step).astype(np.float32)
print(eps_step_vector)

Configurando la fuerza del ataque (eps) por característica...
[ 0.33812526  0.16436736  0.          0.          0.         48.22424
  7.1204505   3.0582452  41.2263     45.8237      0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.        ]
[8.4531315e-02 4.1091841e-02 1.0000000e-06 1.0000000e-06 1.0000000e-06
 1.2056060e+01 1.7801126e+00 7.6456130e-01 1.0306575e+01 1.1455925e+01
 1.0000000e-06 1.0000000e-06 1.0000000e-06 1.0000000e-06 1.0000000e-06
 1.0000000e-06 1.0000000e-06 1.0000000e-06 1.0000000e-06 1.0000000e-06
 1.0000000e-06 1.0000000e-06]


In [25]:
# =======================================================
# FASE 3. LANZAR ATAQUE PGD CON RESTRICCIONES TABULARES
# =======================================================

print("Lanzando ataque PGD...")

ataque_pgd = ProjectedGradientDescent(
    estimator=clasificador_art_mlp,
    eps=eps_vector,
    eps_step=eps_step_vector,
    max_iter=20,
    batch_size=180
)

# Atacamos el mismo conjunto y en el mismo espacio que usa el MLP
x_test_mlp_attack = X_test_scaled_mlp.astype(np.float32)

# Generamos el dataset adversario sin modificar columnas one-hot
x_test_adversario = ataque_pgd.generate(
    x=x_test_mlp_attack,
    mask=attack_mask
)
print("¡Tráfico adversario generado con éxito!")

Lanzando ataque PGD...


PGD - Random Initializations:   0%|          | 0/1 [00:00<?, ?it/s]

PGD - Random Initializations: 100%|██████████| 1/1 [00:00<00:00,  6.60it/s]

¡Tráfico adversario generado con éxito!


In [26]:
# ========================================================
# FASE 4. LANZAR ATAQUE FGSM
# ========================================================

print("Lanzando ataque FGSM...")

ataque_fgsm = FastGradientMethod(
    estimator=clasificador_art_mlp,
    eps=eps_vector,
    eps_step=eps_step_vector,
    batch_size=128,
)

# Usamos el mismo conjunto escalado que consume el MLP
# y bloqueamos también las columnas one-hot
x_test_adversario_fgsm = ataque_fgsm.generate(
    x=x_test_mlp_attack,
    mask=attack_mask
)
print("¡Tráfico adversario FGSM generado con éxito!")

Lanzando ataque FGSM...
¡Tráfico adversario FGSM generado con éxito!


In [27]:
# ========================================================
# FASE 5. COMPARAR ACCURACY Y F1 EN LIMPIO, PGD Y FGSM
# ========================================================

y_test_pred_mlp_clean = mlp_predict_labels(X_test_scaled_mlp)
mlp_acc_clean = accuracy_score(y_test_np, y_test_pred_mlp_clean)
mlp_f1_clean = f1_score(y_test_np, y_test_pred_mlp_clean)

print(f"\nMLP Accuracy en test limpio: {mlp_acc_clean:.4f}")
print(f"MLP F1-score en test limpio: {mlp_f1_clean:.4f}")

y_test_pred_mlp_adv = mlp_predict_labels(x_test_adversario)
mlp_acc_adv = accuracy_score(y_test_np, y_test_pred_mlp_adv)
mlp_f1_adv = f1_score(y_test_np, y_test_pred_mlp_adv)

print(f"\nMLP Accuracy en test adversario PGD: {mlp_acc_adv:.4f}")
print(f"MLP F1-score en test adversario PGD: {mlp_f1_adv:.4f}")

y_test_pred_mlp_fgsm = mlp_predict_labels(x_test_adversario_fgsm)
mlp_acc_fgsm = accuracy_score(y_test_np, y_test_pred_mlp_fgsm)
mlp_f1_fgsm = f1_score(y_test_np, y_test_pred_mlp_fgsm)

print(f"\nMLP Accuracy en test adversario FGSM: {mlp_acc_fgsm:.4f}")
print(f"MLP F1-score en test adversario FGSM: {mlp_f1_fgsm:.4f}")


MLP Accuracy en test limpio: 0.9955
MLP F1-score en test limpio: 0.9962

MLP Accuracy en test adversario PGD: 0.4059
MLP F1-score en test adversario PGD: 0.0218

MLP Accuracy en test adversario FGSM: 0.1373
MLP F1-score en test adversario FGSM: 0.0381


In [28]:
# RF en test limpio
y_test_pred_rf_clean = rf_model.predict(X_test_np)

rf_acc_clean = accuracy_score(y_test_np, y_test_pred_rf_clean)
rf_f1_clean = f1_score(y_test_np, y_test_pred_rf_clean)

print(f"RF Accuracy en test limpio: {rf_acc_clean:.4f}")
print(f"RF F1-score en test limpio: {rf_f1_clean:.4f}")

# ==========================================
# Transferencia desde el MLP al RF
# ==========================================

# Adversarios generados sobre el MLP, convertidos al espacio original
x_test_adversario_raw = mlp_scaler.inverse_transform(x_test_adversario)

# RF sobre tráfico adversario transferido desde el MLP
y_test_pred_rf_adv = rf_model.predict(x_test_adversario_raw)

rf_acc_adv = accuracy_score(y_test_np, y_test_pred_rf_adv)
rf_f1_adv = f1_score(y_test_np, y_test_pred_rf_adv)

print(f"\nRF Accuracy en test adversario transferido: {rf_acc_adv:.4f}")
print(f"RF F1-score en test adversario transferido: {rf_f1_adv:.4f}")


RF Accuracy en test limpio: 0.9996
RF F1-score en test limpio: 0.9997

RF Accuracy en test adversario transferido: 0.8220
RF F1-score en test adversario transferido: 0.8255
